# 02. Auto Labeling with MobileSAM

## AL-03. Input Image Preparation (원본 이미지 준비)

### Objective (목적)
CropAndWeed 데이터셋을 대상으로 MobileSAM 기반 AI-Assisted Labeling(인공지능 보조 라벨링)을 수행한다.

### Workflow (작업 흐름)
1. Input Image (원본 이미지) 불러오기
2. Point Prompt (점 프롬프트) 선정
3. Segmentation Mask (분할 마스크) 자동 생성
4. Visualization (시각화) 및 결과 저장 
5. Reference Mask (정답 마스크)와 비교 평가

### Evaluation (평가)
- IoU (Intersection over Union, 교집합/합집합 비율)
- Dice Coefficient (다이스 유사도)

※ MobileSAM은 기본적으로 클래스 구분 없이 영역을 분할하므로 Crop/Weed 분류 성능은 별도로 평가한다.

In [ ]:
import sys
import torch

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())


In [ ]:
# AL-03C-01: Input Image Loading

from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

# Project Root (프로젝트 루트)
PROJECT_ROOT = Path (
    r"C:\Projects\data-labeling-portfolio\03_smart_agriculture"
)

# Input Image (원본 이미지)
IMAGE_PATH = PROJECT_ROOT / "data/raw/images/ave-0000-0001.jpg"

# Load Image (이미지 불러오기)
image = Image.open(IMAGE_PATH).convert("RGB")
image_np = np.array(image)

print("Image Name:", IMAGE_PATH.name)
print("Image Size:", image.size)
print("Image Array Shape:", image_np.shape)

# Visualization (원본 이미지 시각화)
plt.figure(figsize=(9, 7))
plt.imshow(image_np)
plt.title("AL-03C | Original CropAndWeed Image")
plt.axis("off")
plt.show()

In [ ]:
# AL-03D-01: Sample Image Review


from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

# Image Directory (이미지 폴더)
IMAGE_DIR = PROJECT_ROOT / "data/raw/images"

# Sample Images (샘플 이미지 9장)
image_files = sorted(IMAGE_DIR.glob("*.jpg"))[:9]

print("Selected Images:", len(image_files))

fig, axes = plt.subplots(3, 3, figsize=(15, 10))

for ax, img_path in zip(axes.flat, image_files):
    img = Image.open(img_path).convert("RGB")
    ax.imshow(img)
    ax.set_title(img_path.name, fontsize=10)
    ax.axis("off")

for ax in axes.flat[len(image_files):]:
    ax.axis("off")

plt.suptitle(
    "AL-03D | CropAndWeed Sample Image Review",
    fontsize=14
)
plt.tight_layout()
plt.show()


In [ ]:

# AL-03E-01: Target Image Selection
TARGET_IMAGE = "ave-0035-0001.jpg"

IMAGE_PATH = PROJECT_ROOT / "data/raw/images" / TARGET_IMAGE

image = Image.open(IMAGE_PATH).convert("RGB")
image_np = np.array(image)

height, width = image_np.shape[:2]

print("Target Image:", TARGET_IMAGE)
print("Image Size:", width, "x", height)

plt.figure(figsize=(12, 7))
plt.imshow(image_np)
plt.title("AL-03E | Target Image for MobileSAM")
plt.xlabel("X coordinate (pixels)")
plt.ylabel("Y coordinate (pixels)")
plt.grid(alpha=0.3)
plt.show()


In [ ]:

# AL-03E-02: MobileSAM Point-Prompted Inference

import torch
import numpy as np
import matplotlib.pyplot as plt
from mobile_sam import sam_model_registry, SamPredictor

# 1. Model Loading (모델 불러오기)
CHECKPOINT_PATH = PROJECT_ROOT / "models/mobile_sam.pt"
device = "cuda" if torch.cuda.is_available() else "cpu"

sam = sam_model_registry["vit_t"](
    checkpoint=str(CHECKPOINT_PATH)
)
sam.to(device=device)
sam.eval()

# 2. Predictor Setup (예측기 설정)
predictor = SamPredictor(sam)
predictor.set_image(image_np)

# 3. Point Prompt (점 프롬프트)
input_point = np.array([[1050, 650]])
input_label = np.array([1])

# 4. Inference (추론)
with torch.inference_mode():
    masks, scores, logits = predictor.predict(
        point_coords=input_point,
        point_labels=input_label,
        multimask_output=True
    )

# 5. Select Best Mask (최고 점수 마스크 선정)
best_idx = int(np.argmax(scores))
pred_mask = masks[best_idx]

print("Device:", device)
print("Generated Masks:", len(masks))
print("Best Mask Index:", best_idx)
print("Predicted Quality Score:", round(float(scores[best_idx]), 4))
print("Mask Shape:", pred_mask.shape)

# 6. Visualization (예측 결과 시각화)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].imshow(image_np)
axes[0].scatter(
    input_point[:, 0], input_point[:, 1],
    c="red", marker="*", s=180
)
axes[0].set_title("Input Image + Point Prompt")

axes[1].imshow(image_np)
axes[1].imshow(
    np.ma.masked_where(~pred_mask, pred_mask),
    cmap="spring",
    alpha=0.5
)
axes[1].set_title("MobileSAM Predicted Mask")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# AL-04-01: Reference Mask Discovery

from pathlib import Path

# Reference Mask Discovery (정답 마스크 검색)
mask_files = [
    p for p in (PROJECT_ROOT / "data").rglob("*")
    if p.is_file()
    and p.suffix.lower() in [".png", ".tif", ".tiff"]
    and "mask" in str(p).lower()
]

print("Total Mask Files:", len(mask_files))

for path in mask_files[:20]:
    print(path.relative_to(PROJECT_ROOT))


In [ ]:
# AL-04-02: Dataset Structure Inspection

from pathlib import Path
from collections import Counter

# AL-04-02: Dataset Structure Inspection
DATA_DIR = PROJECT_ROOT / "data"

# 모든 파일 검색
all_files = [p for p in DATA_DIR.rglob("*") if p.is_file()]

# 파일 확장자별 개수
extension_counts = Counter(
    p.suffix.lower() for p in all_files
)

print("=== Total Files ===")
print(len(all_files))

print("\n=== File Extensions ===")
for ext, count in extension_counts.most_common():
    print(ext or "(no extension)", ":", count)

# 현재 실험 이미지와 동일한 이름의 파일 검색
print("\n=== Target Image Related Files ===")
target_stem = "ave-0035-0001"

for p in all_files:
    if target_stem in p.stem.lower():
        print(p.relative_to(PROJECT_ROOT))

# data 폴더의 하위 디렉터리 확인
print("\n=== Data Subdirectories ===")
for p in sorted(DATA_DIR.iterdir()):
    if p.is_dir():
        print(p.relative_to(PROJECT_ROOT))


In [ ]:

# AL-04-03: Reference Mask Inspection

from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

# Reference Mask Path (정답 마스크 경로)
REFERENCE_PATH = (
    PROJECT_ROOT
    / "data/raw/labelIds/CropAndWeed/ave-0035-0001.png"
)

# Load Reference Mask (정답 마스크 불러오기)
reference_image = Image.open(REFERENCE_PATH)
reference_mask = np.array(reference_image)

print("=== Reference Mask Information ===")
print("File:", REFERENCE_PATH.name)
print("Image Mode:", reference_image.mode)
print("Mask Shape:", reference_mask.shape)
print("Mask Data Type:", reference_mask.dtype)

# Label ID Distribution (라벨 ID 분포)
label_ids, counts = np.unique(
    reference_mask, return_counts=True
)

print("\n=== Label ID Distribution ===")
for label_id, count in zip(label_ids, counts):
    print(f"Label ID {label_id}: {count:,} pixels")

# Visualization (정답 마스크 시각화)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].imshow(image_np)
axes[0].set_title("Original Image")

axes[1].imshow(reference_mask, cmap="nipy_spectral")
axes[1].set_title("Reference Label IDs")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

# AL-04-04: Segmentation Quality Metrics
import numpy as np

# Reference Binary Mask (정답 이진 마스크)
gt_mask = (reference_mask == 21)

# Predicted Binary Mask (예측 이진 마스크)
prediction = pred_mask.astype(bool)

# Shape Validation (크기 검증)
assert gt_mask.shape == prediction.shape, "Mask shape mismatch"

# Pixel-level Comparison (픽셀 단위 비교)
intersection = np.logical_and(gt_mask, prediction).sum()
union = np.logical_or(gt_mask, prediction).sum()

gt_pixels = gt_mask.sum()
pred_pixels = prediction.sum()

# IoU and Dice
iou = intersection / union if union > 0 else 1.0

dice = (
    2 * intersection / (gt_pixels + pred_pixels)
    if (gt_pixels + pred_pixels) > 0 else 1.0
)

print("=== AL-04 Segmentation Quality Evaluation ===")
print("Image:", TARGET_IMAGE)
print("Reference Label ID:", 21)
print("Ground Truth Pixels:", gt_pixels)
print("Predicted Pixels:", pred_pixels)
print("Intersection Pixels:", intersection)
print("Union Pixels:", union)
print(f"IoU Score: {iou:.4f}")
print(f"Dice Score: {dice:.4f}")


In [ ]:

# AL-04-05: Pixel-Level Error Visualization
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# Pixel-level Error Masks (픽셀 단위 오류 분류)
tp_mask = gt_mask & prediction
fp_mask = ~gt_mask & prediction
fn_mask = gt_mask & ~prediction

# Error Map (오류 지도)
error_map = np.zeros(
    (*gt_mask.shape, 4), dtype=np.float32
)

# TP: Green / FP: Red / FN: Blue
error_map[tp_mask] = [0, 1, 0, 0.55]
error_map[fp_mask] = [1, 0, 0, 0.85]
error_map[fn_mask] = [0, 0.4, 1, 0.85]

# Visualization (시각화)
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

axes[0].imshow(image_np)
axes[0].set_title("Original Image")

axes[1].imshow(image_np)
axes[1].imshow(error_map)
axes[1].set_title("MobileSAM | Pixel-Level Error Map")

for ax in axes:
    ax.axis("off")

legend_elements = [
    Patch(facecolor="lime", label="TP - Correct"),
    Patch(facecolor="red", label="FP - Over-segmentation"),
    Patch(facecolor="dodgerblue", label="FN - Missed")
]

axes[1].legend(
    handles=legend_elements,
    loc="upper right",
    fontsize=9
)

plt.tight_layout()
plt.show()

print("=== Error Summary ===")
print("True Positive:", int(tp_mask.sum()))
print("False Positive:", int(fp_mask.sum()))
print("False Negative:", int(fn_mask.sum()))


In [ ]:

# AL-05-01: Export Readiness Check
from pathlib import Path
import numpy as np

required_vars = [
    "PROJECT_ROOT",
    "TARGET_IMAGE",
    "image_np",
    "prediction",
    "gt_mask",
    "iou",
    "dice",
    "scores",
    "best_idx"
]

print("=== AL-05 Export Readiness Check ===")

missing = [
    name for name in required_vars
    if name not in globals()
]

for name in required_vars:
    status = "MISSING" if name in missing else "OK"
    print(f"{name}: {status}")

if missing:
    print("\nMissing variables:", missing)
    print("Please rerun the previous notebook cells.")
else:
    assert image_np.shape[:2] == prediction.shape
    assert prediction.shape == gt_mask.shape

    print("\n=== Data Validation ===")
    print("Target Image:", TARGET_IMAGE)
    print("Image Shape:", image_np.shape)
    print("Mask Shape:", prediction.shape)
    print(f"IoU: {iou:.4f}")
    print(f"Dice: {dice:.4f}")
    print("\nAL-05 Export Ready")


In [ ]:

# AL-05-02: Export Prediction and Evaluation Results

from pathlib import Path
from PIL import Image
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# 1. Output Directory (결과 저장 경로)
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "auto_labeling"
MASK_DIR = OUTPUT_DIR / "masks"
FIGURE_DIR = OUTPUT_DIR / "figures"

MASK_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

image_stem = Path(TARGET_IMAGE).stem

# 2. Save Predicted Mask (예측 마스크 저장)
mask_path = MASK_DIR / f"{image_stem}_pred_mask.png"

Image.fromarray(
    prediction.astype(np.uint8) * 255
).save(mask_path)

# 3. Rebuild Pixel-Level Error Map (오류 지도 재생성)
tp_mask = gt_mask & prediction
fp_mask = ~gt_mask & prediction
fn_mask = gt_mask & ~prediction

error_map = np.zeros(
    (*gt_mask.shape, 4), dtype=np.float32
)

error_map[tp_mask] = [0, 1, 0, 0.55]
error_map[fp_mask] = [1, 0, 0, 0.85]
error_map[fn_mask] = [0, 0.4, 1, 0.85]

fig_export, axes = plt.subplots(1, 2, figsize=(15, 6))

axes[0].imshow(image_np)
axes[0].set_title("Original Image")

axes[1].imshow(image_np)
axes[1].imshow(error_map)
axes[1].set_title("MobileSAM | Pixel-Level Error Map")

for ax in axes:
    ax.axis("off")

axes[1].legend(
    handles=[
        Patch(facecolor="lime", label="TP - Correct"),
        Patch(facecolor="red", label="FP - Over-segmentation"),
        Patch(facecolor="dodgerblue", label="FN - Missed")
    ],
    loc="upper right",
    fontsize=9
)

fig_export.tight_layout()

figure_path = FIGURE_DIR / f"{image_stem}_error_map.png"
fig_export.savefig(
    figure_path,
    dpi=150,
    bbox_inches="tight"
)

plt.show()
plt.close(fig_export)

# 4. Save Evaluation Metrics (평가 지표 저장)
metrics_df = pd.DataFrame([{
    "image_name": TARGET_IMAGE,
    "model": "MobileSAM_vit_t",
    "prompt_type": "positive_point",
    "prompt_x": 1050,
    "prompt_y": 650,
    "reference_label_id": 21,
    "iou": float(iou),
    "dice": float(dice),
    "tp": int(tp_mask.sum()),
    "fp": int(fp_mask.sum()),
    "fn": int(fn_mask.sum()),
    "predicted_quality_score": float(scores[best_idx])
}])

metrics_path = OUTPUT_DIR / "mobile_sam_pilot_metrics.csv"

metrics_df.to_csv(
    metrics_path,
    index=False,
    encoding="utf-8-sig"
)

# 5. Export Summary (저장 결과 확인)
print("=== AL-05 Export Summary ===")

for path in [mask_path, figure_path, metrics_path]:
    print(
        path.relative_to(PROJECT_ROOT),
        "| Exists:", path.exists(),
        "| Size:", path.stat().st_size, "bytes"
    )

print("\n=== Saved Metrics ===")
print(metrics_df.to_string(index=False))


In [ ]:

# AL-05-03: Export Validation

from PIL import Image
import numpy as np
import pandas as pd

# 1. Reload Exported Files (저장 파일 다시 불러오기)
saved_mask = np.array(Image.open(mask_path))
saved_metrics = pd.read_csv(metrics_path)

# 2. Validate Mask (마스크 검증)
assert saved_mask.shape == prediction.shape, \
    "Mask shape mismatch"

assert set(np.unique(saved_mask)).issubset({0, 255}), \
    "Unexpected mask pixel values"

assert np.array_equal(
    saved_mask > 0,
    prediction.astype(bool)
), "Saved mask differs from original prediction"

# 3. Validate Metrics (평가 지표 검증)
assert len(saved_metrics) == 1, \
    "Unexpected number of CSV rows"

row = saved_metrics.iloc[0]

assert row["image_name"] == TARGET_IMAGE
assert np.isclose(row["iou"], iou)
assert np.isclose(row["dice"], dice)

assert int(row["tp"]) == int(tp_mask.sum())
assert int(row["fp"]) == int(fp_mask.sum())
assert int(row["fn"]) == int(fn_mask.sum())

# 4. Validate Error Map File (오류 지도 파일 검증)
with Image.open(figure_path) as saved_figure:
    saved_figure.verify()

# 5. Validation Summary (검증 결과)
print("=== AL-05 Export Validation ===")
print("Mask Shape:", saved_mask.shape)
print("Mask Pixel Values:", np.unique(saved_mask))
print("CSV Rows:", len(saved_metrics))
print("IoU:", round(float(row["iou"]), 6))
print("Dice:", round(float(row["dice"]), 6))
print("TP / FP / FN:",
      int(row["tp"]),
      int(row["fp"]),
      int(row["fn"]))

print("\nAL-05 Export Validation: PASS")


In [ ]:

# AL-06-01: Multi-Image Dataset Validation

from pathlib import Path
import pandas as pd

pilot_images = [
    "ave-0035-0001.jpg",
    "ave-0035-0002.jpg",
    "ave-0035-0003.jpg",
    "ave-0007-0007.jpg",
    "ave-0032-0006.jpg"
]

records = []

for filename in pilot_images:
    stem = Path(filename).stem

    image_path = PROJECT_ROOT / "data/raw/images" / filename
    label_path = (
        PROJECT_ROOT
        / "data/raw/labelIds/CropAndWeed"
        / f"{stem}.png"
    )

    records.append({
        "image_name": filename,
        "image_exists": image_path.exists(),
        "label_exists": label_path.exists()
    })

pilot_check_df = pd.DataFrame(records)

print("=== AL-06 Multi-Image Dataset Check ===")
print(pilot_check_df.to_string(index=False))

print(
    "\nAll Files Available:",
    pilot_check_df[
        ["image_exists", "label_exists"]
    ].all().all()
)


In [ ]:

# AL-06-02: Multi-Image Reference Label Inspection

from pathlib import Path
from PIL import Image
import numpy as np
import pandas as pd

label_records = []

for filename in pilot_images:

    stem = Path(filename).stem

    label_path = (
        PROJECT_ROOT
        / "data/raw/labelIds/CropAndWeed"
        / f"{stem}.png"
    )

    # Load Reference Label
    label_img = Image.open(label_path)
    label_array = np.array(label_img)

    # Unique Label IDs
    label_ids, counts = np.unique(
        label_array,
        return_counts=True
    )

    for label_id, count in zip(label_ids, counts):

        label_records.append({
            "image_name": filename,
            "label_id": int(label_id),
            "pixel_count": int(count),
            "pixel_ratio": round(
                count / label_array.size * 100, 4
            )
        })

# Create Summary DataFrame
label_summary_df = pd.DataFrame(label_records)

print("=== AL-06 Reference Label Summary ===")

print(
    label_summary_df.to_string(index=False)
)

# Non-background Label IDs
print("\n=== Non-Background Label IDs ===")

non_background_df = label_summary_df[
    label_summary_df["label_id"] != 0
]

print(
    non_background_df.to_string(index=False)
)


In [ ]:

# AL-06-03-01: Label Metadata Discovery

from pathlib import Path

# Metadata 후보 파일 검색
metadata_files = [
    p for p in PROJECT_ROOT.rglob("*")
    if p.is_file()
    and p.suffix.lower() in [".csv", ".json", ".yaml", ".yml", ".txt"]
    and any(
        keyword in p.name.lower()
        for keyword in [
            "label", "class", "mapping",
            "metadata", "description"
        ]
    )
    and ".venv" not in p.parts
]

print("=== Label Metadata Candidates ===")
print("Total Candidates:", len(metadata_files))

for path in metadata_files[:30]:
    print(path.relative_to(PROJECT_ROOT))


In [ ]:

# AL-06-04-01: Multi-Image Visualization
# Positive Point Prompt 선정을 위한 원본 이미지 확인

from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(
    len(pilot_images), 1,
    figsize=(14, 25)
)

for ax, filename in zip(axes, pilot_images):

    image_path = (
        PROJECT_ROOT / "data/raw/images" / filename
    )

    image = np.array(
        Image.open(image_path).convert("RGB")
    )

    height, width = image.shape[:2]

    ax.imshow(image)

    ax.set_title(
        f"{filename} | Width: {width}, Height: {height}",
        fontsize=12
    )

    # Coordinate grid
    ax.set_xticks(np.arange(0, width + 1, 200))
    ax.set_yticks(np.arange(0, height + 1, 200))

    ax.grid(
        color="white",
        linestyle="--",
        linewidth=0.7,
        alpha=0.6
    )

    ax.set_xlim(0, width)
    ax.set_ylim(height, 0)

plt.tight_layout()
plt.show()


In [ ]:

# AL-06-05-00: Point Prompt Configuration

pilot_prompts = {
    "ave-0035-0001.jpg": [(1050, 650)],
    "ave-0035-0002.jpg": [(350, 750), (1050, 720)],
    "ave-0035-0003.jpg": [(930, 550)],
    "ave-0007-0007.jpg": [(990, 590), (1700, 180)],
    "ave-0032-0006.jpg": [(1320, 700)]
}

# Validation
assert all(
    filename in pilot_prompts
    for filename in pilot_images
), "Missing Point Prompt"

print("=== Point Prompt Configuration ===")

for filename in pilot_images:
    print(f"{filename}: {pilot_prompts[filename]}")

print("\nPoint Prompt Configuration: PASS")


In [ ]:

# AL-06-05-01: Multi-Image MobileSAM Inference

import numpy as np
import pandas as pd
from PIL import Image
from mobile_sam import SamPredictor

# 기존 AL-02에서 로드한 MobileSAM 모델 사용
predictor = SamPredictor(sam)

pilot_predictions = {}
inference_records = []

for filename in pilot_images:

    image_path = (
        PROJECT_ROOT / "data/raw/images" / filename
    )

    image_np_current = np.array(
        Image.open(image_path).convert("RGB")
    )

    height, width = image_np_current.shape[:2]

    # 이미지별 Embedding 계산
    predictor.set_image(image_np_current)

    # 이미지 전체를 위한 Binary Mask
    combined_mask = np.zeros(
        (height, width),
        dtype=bool
    )

    points = pilot_prompts[filename]
    point_masks = []
    point_scores = []

    for point_idx, (x, y) in enumerate(points, start=1):

        input_point = np.array(
            [[x, y]],
            dtype=np.float32
        )

        input_label = np.array([1])

        masks, scores, logits = predictor.predict(
            point_coords=input_point,
            point_labels=input_label,
            multimask_output=True
        )

        # 예측 점수가 가장 높은 Mask 선택
        best_idx = int(np.argmax(scores))

        selected_mask = masks[best_idx].astype(bool)
        selected_score = float(scores[best_idx])

        # 이미지 전체 Mask에 통합
        combined_mask |= selected_mask

        point_masks.append(selected_mask)
        point_scores.append(selected_score)

        inference_records.append({
            "image_name": filename,
            "point_id": point_idx,
            "point_x": x,
            "point_y": y,
            "predicted_score": selected_score,
            "mask_pixels": int(selected_mask.sum())
        })

    pilot_predictions[filename] = {
        "image": image_np_current,
        "combined_mask": combined_mask,
        "point_masks": point_masks,
        "point_scores": point_scores,
        "points": points
    }

    print(
        f"[DONE] {filename} | "
        f"Points: {len(points)} | "
        f"Mask Pixels: {combined_mask.sum():,}"
    )

inference_df = pd.DataFrame(inference_records)

print("\n=== AL-06 Multi-Image Inference Summary ===")
print(inference_df.to_string(index=False))

print("\nProcessed Images:", len(pilot_predictions))
print("Processed Points:", len(inference_df))


In [ ]:

# AL-06-05-02: Multi-Image Prediction Visualization

import matplotlib.pyplot as plt

fig, axes = plt.subplots(
    len(pilot_images), 2,
    figsize=(16, 24)
)

for row_idx, filename in enumerate(pilot_images):

    result = pilot_predictions[filename]

    image = result["image"]
    combined_mask = result["combined_mask"]
    points = np.array(result["points"])

    # Original Image + Point Prompt
    axes[row_idx, 0].imshow(image)

    axes[row_idx, 0].scatter(
        points[:, 0],
        points[:, 1],
        c="red",
        marker="*",
        s=130,
        edgecolors="white"
    )

    axes[row_idx, 0].set_title(
        f"{filename} | Input Prompts"
    )

    # Prediction Overlay
    axes[row_idx, 1].imshow(image)

    overlay = np.ma.masked_where(
        ~combined_mask,
        combined_mask
    )

    axes[row_idx, 1].imshow(
        overlay,
        cmap="Greens",
        alpha=0.5,
        vmin=0,
        vmax=1
    )

    axes[row_idx, 1].set_title(
        f"MobileSAM Prediction | "
        f"{combined_mask.sum():,} pixels"
    )

    for ax in axes[row_idx]:
        ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

# AL-06-06-01: Multi-Image Segmentation Evaluation
# 임시 가정:
# 0 = Background
# 255 = Ignore (검증 전 임시 제외)
# 그 외 ID = Foreground

import numpy as np
import pandas as pd
from PIL import Image

evaluation_records = []

for filename in pilot_images:

    stem = filename.replace(".jpg", "")

    label_path = (
        PROJECT_ROOT
        / "data/raw/labelIds/CropAndWeed"
        / f"{stem}.png"
    )

    reference_mask = np.array(
        Image.open(label_path)
    )

    prediction_mask = pilot_predictions[
        filename
    ]["combined_mask"].astype(bool)

    assert (
        reference_mask.shape == prediction_mask.shape
    ), f"Shape mismatch: {filename}"

    # Temporary binary foreground definition
    valid_pixels = reference_mask != 255

    gt_binary = (
        (reference_mask != 0) &
        valid_pixels
    )

    # Evaluate only valid pixels
    tp = int(np.sum(
        prediction_mask & gt_binary & valid_pixels
    ))

    fp = int(np.sum(
        prediction_mask & ~gt_binary & valid_pixels
    ))

    fn = int(np.sum(
        ~prediction_mask & gt_binary & valid_pixels
    ))

    iou = (
        tp / (tp + fp + fn)
        if (tp + fp + fn) > 0 else np.nan
    )

    dice = (
        2 * tp / (2 * tp + fp + fn)
        if (2 * tp + fp + fn) > 0 else np.nan
    )

    precision = (
        tp / (tp + fp)
        if (tp + fp) > 0 else np.nan
    )

    recall = (
        tp / (tp + fn)
        if (tp + fn) > 0 else np.nan
    )

    evaluation_records.append({
        "image_name": filename,
        "gt_pixels": int(gt_binary.sum()),
        "pred_pixels_valid": int(
            np.sum(prediction_mask & valid_pixels)
        ),
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "iou": round(iou, 4),
        "dice": round(dice, 4),
        "precision": round(precision, 4),
        "recall": round(recall, 4)
    })

evaluation_df = pd.DataFrame(evaluation_records)

print("=== AL-06 Multi-Image Evaluation ===")
print(evaluation_df.to_string(index=False))

print("\n=== Average Metrics ===")
print(
    evaluation_df[
        ["iou", "dice", "precision", "recall"]
    ].mean().round(4)
)

print("\nEvaluation Status: PROVISIONAL")
print("Label ID mapping requires verification.")


In [ ]:

# AL-06-07-01: Point Prompt Improvement Setup

from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

target_filename = "ave-0032-0006.jpg"

image_path = (
    PROJECT_ROOT
    / "data/raw/images"
    / target_filename
)

target_image = np.array(
    Image.open(image_path).convert("RGB")
)

# 실험별 프롬프트 좌표
prompt_experiments = {
    "Baseline": [(1320, 700)],
    "Experiment A": [(1320, 560)],
    "Experiment B": [(1320, 560), (1320, 700)]
}

fig, axes = plt.subplots(
    1, 3,
    figsize=(18, 6)
)

for ax, (experiment, points) in zip(
    axes,
    prompt_experiments.items()
):

    ax.imshow(target_image)

    for idx, (x, y) in enumerate(points, start=1):

        ax.scatter(
            x, y,
            c="red",
            s=180,
            marker="*",
            edgecolors="white",
            linewidths=1.2
        )

        ax.annotate(
            f"P{idx}",
            (x, y),
            xytext=(10, -12),
            textcoords="offset points",
            color="yellow",
            fontsize=11,
            fontweight="bold"
        )

    ax.set_title(experiment, fontsize=13)

    # 식물 주변 확대
    ax.set_xlim(1150, 1500)
    ax.set_ylim(900, 400)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("=== AL-06-07 Prompt Experiments ===")

for experiment, points in prompt_experiments.items():
    print(f"{experiment}: {points}")


In [ ]:

# AL-06-07-02: MobileSAM Prompt Comparison

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from mobile_sam import SamPredictor

# 실험 대상 이미지
target_filename = "ave-0032-0006.jpg"

# 기존 AL-06-07-01의 prompt_experiments 사용
predictor_compare = SamPredictor(sam)
predictor_compare.set_image(target_image)

# 기존 평가와 동일한 잠정 GT 정의
label_path = (
    PROJECT_ROOT
    / "data/raw/labelIds/CropAndWeed"
    / target_filename.replace(".jpg", ".png")
)

reference_mask = np.array(Image.open(label_path))

valid_pixels = reference_mask != 255
gt_binary = (reference_mask != 0) & valid_pixels

experiment_results = {}
comparison_records = []

for experiment_name, points in prompt_experiments.items():

    combined_mask = np.zeros(
        target_image.shape[:2],
        dtype=bool
    )

    point_scores = []

    # 각 Point를 독립적으로 추론한 후 통합
    for x, y in points:

        masks, scores, _ = predictor_compare.predict(
            point_coords=np.array([[x, y]]),
            point_labels=np.array([1]),
            multimask_output=True
        )

        best_idx = int(np.argmax(scores))

        selected_mask = masks[best_idx].astype(bool)
        combined_mask |= selected_mask

        point_scores.append(float(scores[best_idx]))

    # 유효 픽셀만 평가
    tp = int(np.sum(
        combined_mask & gt_binary & valid_pixels
    ))
    fp = int(np.sum(
        combined_mask & ~gt_binary & valid_pixels
    ))
    fn = int(np.sum(
        ~combined_mask & gt_binary & valid_pixels
    ))

    iou = tp / (tp + fp + fn) if (tp + fp + fn) else np.nan
    dice = (
        2 * tp / (2 * tp + fp + fn)
        if (2 * tp + fp + fn) else np.nan
    )
    precision = tp / (tp + fp) if (tp + fp) else np.nan
    recall = tp / (tp + fn) if (tp + fn) else np.nan

    experiment_results[experiment_name] = {
        "mask": combined_mask,
        "points": points,
        "scores": point_scores
    }

    comparison_records.append({
        "experiment": experiment_name,
        "point_count": len(points),
        "pred_pixels": int(combined_mask.sum()),
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "iou": round(iou, 4),
        "dice": round(dice, 4),
        "precision": round(precision, 4),
        "recall": round(recall, 4)
    })

comparison_df = pd.DataFrame(comparison_records)

print("=== AL-06-07 Prompt Comparison ===")
print(comparison_df.to_string(index=False))

# Baseline 대비 변화
baseline_iou = comparison_df.loc[
    comparison_df["experiment"] == "Baseline", "iou"
].iloc[0]

comparison_df["iou_delta"] = (
    comparison_df["iou"] - baseline_iou
).round(4)

print("\n=== IoU Improvement vs Baseline ===")
print(
    comparison_df[
        ["experiment", "iou", "iou_delta"]
    ].to_string(index=False)
)

# 세 실험 결과 시각화
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, (name, result) in zip(
    axes, experiment_results.items()
):
    ax.imshow(target_image)

    mask_overlay = np.ma.masked_where(
        ~result["mask"], result["mask"]
    )

    ax.imshow(
        mask_overlay,
        cmap="Greens",
        alpha=0.5,
        vmin=0,
        vmax=1
    )

    points = np.array(result["points"])
    ax.scatter(
        points[:, 0],
        points[:, 1],
        c="red",
        marker="*",
        s=140,
        edgecolors="white"
    )

    score = comparison_df.loc[
        comparison_df["experiment"] == name, "iou"
    ].iloc[0]

    ax.set_title(f"{name} | IoU: {score:.4f}")
    ax.set_xlim(1150, 1500)
    ax.set_ylim(900, 400)
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

print("\nEvaluation Status: PROVISIONAL")
print("Label ID mapping requires verification.")


In [ ]:

# AL-06-08-01: Export Multi-Image Pilot Results

from pathlib import Path
from PIL import Image
import numpy as np

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "auto_labeling"
MASK_DIR = OUTPUT_DIR / "masks"
FIGURE_DIR = OUTPUT_DIR / "figures"

MASK_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# 1. Evaluation Metrics 저장
metrics_path_multi = (
    OUTPUT_DIR / "mobile_sam_multi_image_metrics.csv"
)

evaluation_df.to_csv(
    metrics_path_multi,
    index=False,
    encoding="utf-8-sig"
)

# 2. Prompt Comparison 저장
comparison_path = (
    OUTPUT_DIR / "mobile_sam_prompt_comparison.csv"
)

comparison_df.to_csv(
    comparison_path,
    index=False,
    encoding="utf-8-sig"
)

# 3. 이미지별 Baseline Prediction Mask 저장
exported_masks = []

for filename in pilot_images:

    stem = Path(filename).stem

    mask = pilot_predictions[
        filename
    ]["combined_mask"]

    mask_path_multi = (
        MASK_DIR / f"{stem}_pred_mask.png"
    )

    Image.fromarray(
        mask.astype(np.uint8) * 255
    ).save(mask_path_multi)

    exported_masks.append(mask_path_multi)

# 4. Experiment B 개선 Mask 별도 저장
improved_mask_path = (
    MASK_DIR / "ave-0032-0006_improved_mask.png"
)

Image.fromarray(
    experiment_results["Experiment B"]["mask"]
    .astype(np.uint8) * 255
).save(improved_mask_path)

print("=== AL-06-08 Export Summary ===")

print("Metrics:", metrics_path_multi.name)
print("Comparison:", comparison_path.name)

print("\nExported Baseline Masks:")
for path in exported_masks:
    print("-", path.name)

print("\nImproved Mask:")
print("-", improved_mask_path.name)

print("\nExport Status: COMPLETE")


In [ ]:

# AL-06-08-02: Export Validation

import pandas as pd
import numpy as np
from PIL import Image

# CSV Validation
saved_metrics = pd.read_csv(metrics_path_multi)
saved_comparison = pd.read_csv(comparison_path)

assert len(saved_metrics) == 5
assert len(saved_comparison) == 3

# Mask Validation
for filename in pilot_images:

    stem = Path(filename).stem

    saved_path = (
        MASK_DIR / f"{stem}_pred_mask.png"
    )

    saved_mask = np.array(
        Image.open(saved_path)
    )

    original_mask = pilot_predictions[
        filename
    ]["combined_mask"]

    assert set(np.unique(saved_mask)).issubset({0, 255})

    assert np.array_equal(
        saved_mask > 0,
        original_mask
    ), f"Mask mismatch: {filename}"

# Improved Mask Validation
saved_improved = np.array(
    Image.open(improved_mask_path)
)

assert np.array_equal(
    saved_improved > 0,
    experiment_results["Experiment B"]["mask"]
)

print("=== AL-06-08 Export Validation ===")
print("Metrics Rows:", len(saved_metrics))
print("Comparison Rows:", len(saved_comparison))
print("Baseline Masks:", len(pilot_images))
print("Improved Mask:", 1)

print("\nAL-06-08 Export Validation: PASS")


In [ ]:

# AL-06-08-03: Generate Pilot QA Summary

mean_metrics = evaluation_df[
    ["iou", "dice", "precision", "recall"]
].mean()

baseline_row = comparison_df[
    comparison_df["experiment"] == "Baseline"
].iloc[0]

improved_row = comparison_df[
    comparison_df["experiment"] == "Experiment B"
].iloc[0]

iou_change = improved_row["iou"] - baseline_row["iou"]

qa_summary = f"""# MobileSAM Multi-Image Pilot QA Summary

## 1. Project Overview

- Project: Smart Agriculture Data Labeling
- Model: MobileSAM (vit_t)
- Method: Point-Prompt AI-Assisted Segmentation
- Pilot Images: {len(pilot_images)}
- Positive Point Prompts: {len(inference_df)}
- Evaluation Status: PROVISIONAL

## 2. Average Performance

- Mean IoU: {mean_metrics['iou']:.4f}
- Mean Dice: {mean_metrics['dice']:.4f}
- Mean Precision: {mean_metrics['precision']:.4f}
- Mean Recall: {mean_metrics['recall']:.4f}

## 3. Key Findings

1. Precision was relatively high across the pilot images.
2. Recall varied substantially between images.
3. Small or partially separated plant regions were
   more difficult to capture with a single point prompt.
4. Prompt placement and prompt count affected segmentation.

## 4. Prompt Improvement Experiment

- Target Image: ave-0032-0006.jpg
- Baseline IoU: {baseline_row['iou']:.4f}
- Improved IoU: {improved_row['iou']:.4f}
- IoU Change: {iou_change:+.4f}

The improved experiment used two independent point prompts
and combined the resulting masks.

## 5. Evaluation Limitations

- Label ID mapping has not been fully verified.
- Label ID 255 was provisionally treated as an ignore region.
- All other nonzero IDs were provisionally treated as foreground.
- Point prompts were manually selected.
- This is an exploratory five-image pilot, not an independent
  benchmark or a fully automatic labeling evaluation.

## 6. Next Steps

1. Verify reference label mapping.
2. Convert predicted masks to polygons.
3. Perform human QA and polygon correction.
4. Validate final annotation exports.
"""

qa_summary_path = (
    OUTPUT_DIR / "mobile_sam_pilot_qa_summary.md"
)

qa_summary_path.write_text(
    qa_summary,
    encoding="utf-8"
)

print("=== AL-06 Pilot QA Summary ===")
print("Saved:", qa_summary_path)
print("\n", qa_summary)


In [ ]:

# AL-06-08-04: Export QA Visualizations

import matplotlib.pyplot as plt
import numpy as np

# 1. Multi-Image Prediction Overview
fig, axes = plt.subplots(
    len(pilot_images), 2,
    figsize=(16, 24)
)

for row_idx, filename in enumerate(pilot_images):

    result = pilot_predictions[filename]
    image = result["image"]
    mask = result["combined_mask"]
    points = np.array(result["points"])

    axes[row_idx, 0].imshow(image)
    axes[row_idx, 0].scatter(
        points[:, 0], points[:, 1],
        c="red", marker="*", s=130,
        edgecolors="white"
    )
    axes[row_idx, 0].set_title(
        f"{filename} | Point Prompts"
    )

    axes[row_idx, 1].imshow(image)
    axes[row_idx, 1].imshow(
        np.ma.masked_where(~mask, mask),
        cmap="Greens",
        alpha=0.5,
        vmin=0,
        vmax=1
    )
    axes[row_idx, 1].set_title(
        f"{filename} | MobileSAM Prediction"
    )

    for ax in axes[row_idx]:
        ax.axis("off")

plt.tight_layout()

overview_path = (
    FIGURE_DIR / "mobile_sam_multi_image_overview.png"
)

fig.savefig(
    overview_path,
    dpi=180,
    bbox_inches="tight"
)
plt.close(fig)

# 2. Prompt Improvement Comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, (name, result) in zip(
    axes, experiment_results.items()
):

    ax.imshow(target_image)

    mask = result["mask"]
    ax.imshow(
        np.ma.masked_where(~mask, mask),
        cmap="Greens",
        alpha=0.5,
        vmin=0,
        vmax=1
    )

    points = np.array(result["points"])

    ax.scatter(
        points[:, 0], points[:, 1],
        c="red", marker="*", s=140,
        edgecolors="white"
    )

    score = comparison_df.loc[
        comparison_df["experiment"] == name, "iou"
    ].iloc[0]

    ax.set_title(f"{name} | IoU: {score:.4f}")
    ax.set_xlim(1150, 1500)
    ax.set_ylim(900, 400)
    ax.grid(alpha=0.2)

plt.tight_layout()

comparison_figure_path = (
    FIGURE_DIR / "mobile_sam_prompt_comparison.png"
)

fig.savefig(
    comparison_figure_path,
    dpi=180,
    bbox_inches="tight"
)
plt.close(fig)

# 3. Export Verification
print("=== AL-06 Visualization Export ===")

for path in [overview_path, comparison_figure_path]:
    print(
        f"{path.name} | "
        f"Exists: {path.exists()} | "
        f"Size: {path.stat().st_size:,} bytes"
    )

assert overview_path.exists()
assert comparison_figure_path.exists()

print("\nAL-06 Visualization Export: PASS")


In [ ]:

# AL-07-01: Single-Image Mask-to-Polygon Conversion

import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# 기존 Baseline 이미지 사용
target_filename = "ave-0035-0001.jpg"

image_path = (
    PROJECT_ROOT / "data/raw/images" / target_filename
)

mask_path = (
    PROJECT_ROOT
    / "outputs/auto_labeling/masks"
    / "ave-0035-0001_pred_mask.png"
)

image = np.array(
    Image.open(image_path).convert("RGB")
)

mask = np.array(
    Image.open(mask_path).convert("L")
)

# Binary Mask 확인
binary_mask = (mask > 0).astype(np.uint8) * 255

# Polygon Contour 추출
contours, hierarchy = cv2.findContours(
    binary_mask,
    cv2.RETR_EXTERNAL,
    cv2.CHAIN_APPROX_SIMPLE
)

# 작은 Noise Contour 제외
MIN_AREA = 100

valid_contours = [
    contour for contour in contours
    if cv2.contourArea(contour) >= MIN_AREA
]

# Polygon 시각화
polygon_image = image.copy()

cv2.drawContours(
    polygon_image,
    valid_contours,
    contourIdx=-1,
    color=(255, 0, 0),
    thickness=3
)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

axes[0].imshow(image)
axes[0].imshow(
    binary_mask,
    cmap="Greens",
    alpha=0.35
)
axes[0].set_title("MobileSAM Binary Mask")

axes[1].imshow(polygon_image)
axes[1].set_title(
    f"Polygon Contours | Count: {len(valid_contours)}"
)

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

print("=== AL-07 Polygon Conversion ===")
print("Original Contours:", len(contours))
print("Valid Contours:", len(valid_contours))
print("Minimum Area:", MIN_AREA)


In [ ]:

# AL-07-02-01: Polygon Reconstruction Quality Check

import cv2
import numpy as np

# 원본 MobileSAM Binary Mask
original_binary = binary_mask > 0

# Polygon을 다시 Mask로 복원
reconstructed_mask = np.zeros_like(
    binary_mask,
    dtype=np.uint8
)

cv2.drawContours(
    reconstructed_mask,
    valid_contours,
    contourIdx=-1,
    color=255,
    thickness=cv2.FILLED
)

reconstructed_binary = reconstructed_mask > 0

# Conversion IoU
intersection = np.logical_and(
    original_binary,
    reconstructed_binary
).sum()

union = np.logical_or(
    original_binary,
    reconstructed_binary
).sum()

conversion_iou = (
    intersection / union if union > 0 else np.nan
)

# Pixel Differences
lost_pixels = np.logical_and(
    original_binary,
    ~reconstructed_binary
).sum()

added_pixels = np.logical_and(
    ~original_binary,
    reconstructed_binary
).sum()

# Vertex Count
vertex_counts = [
    len(contour)
    for contour in valid_contours
]

print("=== AL-07 Polygon Quality Validation ===")

print("Original Mask Pixels:", int(original_binary.sum()))
print("Reconstructed Mask Pixels:", int(reconstructed_binary.sum()))

print("Conversion IoU:", round(conversion_iou, 6))

print("Lost Pixels:", int(lost_pixels))
print("Added Pixels:", int(added_pixels))

print("Polygon Count:", len(valid_contours))
print("Vertex Counts:", vertex_counts)


In [ ]:

# AL-07-03-01: Polygon Simplification Experiment

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# AL-07-01에서 생성한 Polygon 사용
original_contour = valid_contours[0]

# Simplification Levels
epsilon_levels = {
    "Original": 0.0,
    "Light": 0.001,
    "Medium": 0.003,
    "Strong": 0.005
}

perimeter = cv2.arcLength(
    original_contour,
    closed=True
)

simplification_records = []
simplified_polygons = {}

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for ax, (level, ratio) in zip(
    axes,
    epsilon_levels.items()
):

    if ratio == 0:
        simplified = original_contour.copy()
    else:
        simplified = cv2.approxPolyDP(
            original_contour,
            epsilon=ratio * perimeter,
            closed=True
        )

    simplified_polygons[level] = simplified

    # Polygon을 Binary Mask로 복원
    reconstructed = np.zeros_like(
        binary_mask,
        dtype=np.uint8
    )

    cv2.fillPoly(
        reconstructed,
        [simplified],
        color=255
    )

    # Original Mask와 비교
    original_bool = binary_mask > 0
    reconstructed_bool = reconstructed > 0

    intersection = np.logical_and(
        original_bool,
        reconstructed_bool
    ).sum()

    union = np.logical_or(
        original_bool,
        reconstructed_bool
    ).sum()

    iou = intersection / union if union else np.nan

    simplification_records.append({
        "level": level,
        "epsilon_ratio": ratio,
        "vertices": len(simplified),
        "conversion_iou": round(iou, 6)
    })

    # Polygon 시각화
    display_image = image.copy()

    cv2.drawContours(
        display_image,
        [simplified],
        -1,
        (255, 0, 0),
        3
    )

    ax.imshow(display_image)
    ax.set_title(
        f"{level}\n"
        f"Vertices: {len(simplified)} | IoU: {iou:.4f}"
    )
    ax.axis("off")

plt.tight_layout()
plt.show()

simplification_df = pd.DataFrame(
    simplification_records
)

print("=== AL-07 Polygon Simplification ===")
print(simplification_df.to_string(index=False))


In [ ]:

# AL-07-05-01: Export Polygon Annotations

import cv2
import json
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

MASK_DIR = PROJECT_ROOT / "outputs/auto_labeling/masks"
ANNOTATION_DIR = (
    PROJECT_ROOT / "outputs/auto_labeling/annotations"
)
ANNOTATION_DIR.mkdir(parents=True, exist_ok=True)

EPSILON_RATIO = 0.001
MIN_AREA = 100

annotations = []
qa_records = []

for filename in pilot_images:

    stem = Path(filename).stem

    image_path = (
        PROJECT_ROOT / "data/raw/images" / filename
    )
    mask_path = MASK_DIR / f"{stem}_pred_mask.png"

    with Image.open(image_path) as img:
        width, height = img.size

    mask = np.array(
        Image.open(mask_path).convert("L")
    )

    assert mask.shape == (height, width)

    binary_mask = (mask > 0).astype(np.uint8) * 255

    contours, _ = cv2.findContours(
        binary_mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    polygons = []
    reconstructed = np.zeros_like(binary_mask)

    for contour in contours:

        if cv2.contourArea(contour) < MIN_AREA:
            continue

        perimeter = cv2.arcLength(contour, True)

        simplified = cv2.approxPolyDP(
            contour,
            EPSILON_RATIO * perimeter,
            True
        )

        if len(simplified) < 3:
            continue

        points = simplified.reshape(-1, 2).tolist()

        polygons.append({
            "polygon_id": len(polygons) + 1,
            "points": points,
            "vertex_count": len(points),
            "source": "MobileSAM"
        })

        cv2.fillPoly(
            reconstructed,
            [simplified],
            color=255
        )

    original_bool = binary_mask > 0
    reconstructed_bool = reconstructed > 0

    intersection = np.logical_and(
        original_bool, reconstructed_bool
    ).sum()

    union = np.logical_or(
        original_bool, reconstructed_bool
    ).sum()

    conversion_iou = (
        float(intersection / union)
        if union > 0 else None
    )

    annotations.append({
        "image_name": filename,
        "width": width,
        "height": height,
        "annotation_type": "polygon",
        "label": "plant_foreground_unverified",
        "qa_status": "pending_human_review",
        "polygons": polygons
    })

    qa_records.append({
        "image_name": filename,
        "polygon_count": len(polygons),
        "total_vertices": sum(
            p["vertex_count"] for p in polygons
        ),
        "conversion_iou": conversion_iou,
        "lost_pixels": int(np.sum(
            original_bool & ~reconstructed_bool
        )),
        "added_pixels": int(np.sum(
            ~original_bool & reconstructed_bool
        ))
    })

# JSON Export
json_path = (
    ANNOTATION_DIR / "mobile_sam_polygon_annotations.json"
)

export_data = {
    "project": "Smart Agriculture Data Labeling",
    "model": "MobileSAM",
    "annotation_method": "point_prompt_ai_assisted",
    "epsilon_ratio": EPSILON_RATIO,
    "min_area": MIN_AREA,
    "status": "draft_pending_human_qa",
    "annotations": annotations
}

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(
        export_data,
        f,
        ensure_ascii=False,
        indent=2
    )

# QA CSV Export
polygon_export_qa_df = pd.DataFrame(qa_records)

csv_path = (
    ANNOTATION_DIR / "mobile_sam_polygon_qa.csv"
)

polygon_export_qa_df.to_csv(
    csv_path,
    index=False,
    encoding="utf-8-sig"
)

print("=== AL-07 Final Annotation Export ===")
print("JSON:", json_path.name)
print("CSV:", csv_path.name)
print("Images:", len(annotations))
print("Total Polygons:", sum(
    len(item["polygons"]) for item in annotations
))
print("\n=== Polygon Export QA ===")
print(polygon_export_qa_df.to_string(index=False))


In [ ]:

# AL-07-05-02: Annotation Export Validation

import json
import pandas as pd

with open(json_path, "r", encoding="utf-8") as f:
    saved_data = json.load(f)

saved_qa = pd.read_csv(csv_path)

assert saved_data["status"] == "draft_pending_human_qa"
assert len(saved_data["annotations"]) == len(pilot_images)
assert len(saved_qa) == len(pilot_images)

for item in saved_data["annotations"]:

    width = item["width"]
    height = item["height"]

    for polygon in item["polygons"]:

        points = polygon["points"]

        assert len(points) >= 3
        assert polygon["vertex_count"] == len(points)

        for x, y in points:
            assert 0 <= x < width
            assert 0 <= y < height

print("=== AL-07 Annotation Validation ===")
print("Images:", len(saved_data["annotations"]))
print("Total Polygons:", sum(
    len(item["polygons"])
    for item in saved_data["annotations"]
))
print("QA Records:", len(saved_qa))
print("Coordinate Validation: PASS")
print("\nAL-07 Annotation Export Validation: PASS")


## AL-08. Human QA & Annotation Correction

### AL-08-01. Annotation QA Visualization

MobileSAM이 생성한 Polygon Annotation을 원본 이미지에 중첩하여 누락 영역, 과잉 영역 및 경계 오류를 시각적으로 검토한다.

In [ ]:

# AL-08-01-01: Annotation QA Visualization
# 저장된 Polygon JSON을 불러와 원본 이미지에 표시

import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from matplotlib.patches import Polygon as MplPolygon

# 1. Annotation JSON 불러오기
annotation_path = (
    PROJECT_ROOT
    / "outputs/auto_labeling/annotations"
    / "mobile_sam_polygon_annotations.json"
)

with open(annotation_path, "r", encoding="utf-8") as f:
    annotation_data = json.load(f)

annotations = annotation_data["annotations"]

# 2. 시각화
fig, axes = plt.subplots(
    len(annotations), 2,
    figsize=(16, 24),
    squeeze=False
)

for row_idx, item in enumerate(annotations):

    filename = item["image_name"]

    image_path = (
        PROJECT_ROOT / "data/raw/images" / filename
    )

    image = np.array(
        Image.open(image_path).convert("RGB")
    )

    # 왼쪽: Original Image
    ax_original = axes[row_idx, 0]
    ax_original.imshow(image)
    ax_original.set_title(
        f"{filename} | Original Image"
    )
    ax_original.axis("off")

    # 오른쪽: Polygon Annotation
    ax_polygon = axes[row_idx, 1]
    ax_polygon.imshow(image)

    for polygon in item["polygons"]:

        points = np.array(
            polygon["points"],
            dtype=np.float32
        )

        patch = MplPolygon(
            points,
            closed=True,
            fill=True,
            facecolor="lime",
            edgecolor="red",
            linewidth=2,
            alpha=0.35
        )

        ax_polygon.add_patch(patch)

        # Polygon ID 표시
        center_x = points[:, 0].mean()
        center_y = points[:, 1].mean()

        ax_polygon.text(
            center_x,
            center_y,
            f"P{polygon['polygon_id']}",
            color="white",
            fontsize=11,
            fontweight="bold",
            ha="center",
            va="center",
            bbox={
                "facecolor": "black",
                "alpha": 0.6,
                "pad": 3
            }
        )

    ax_polygon.set_title(
        f"{filename} | "
        f"Polygon Count: {len(item['polygons'])}"
    )
    ax_polygon.axis("off")

plt.tight_layout()
plt.show()

print("=== AL-08 Annotation QA Visualization ===")
print("Images Reviewed:", len(annotations))
print(
    "Total Polygons:",
    sum(len(item["polygons"]) for item in annotations)
)
print("QA Visualization: COMPLETE")


In [ ]:

# AL-08-02-01: Human QA Review Log
# 시각적 검토에 따른 잠정 QA 기록

import pandas as pd

qa_review_records = [
    {
        "image_name": "ave-0035-0001.jpg",
        "qa_status": "Pass Candidate",
        "error_type": "None Observed",
        "severity": "Low",
        "review_note": "Major plant region covered; detailed boundary QA pending."
    },
    {
        "image_name": "ave-0035-0002.jpg",
        "qa_status": "Pass Candidate",
        "error_type": "None Observed",
        "severity": "Low",
        "review_note": "Both main plants covered; detailed boundary QA pending."
    },
    {
        "image_name": "ave-0035-0003.jpg",
        "qa_status": "Review",
        "error_type": "Missing Region",
        "severity": "Medium",
        "review_note": "Possible missing leaves and small surrounding plants."
    },
    {
        "image_name": "ave-0007-0007.jpg",
        "qa_status": "Review",
        "error_type": "Boundary Error",
        "severity": "Medium",
        "review_note": "Inspect small leaves and polygon boundaries at higher zoom."
    },
    {
        "image_name": "ave-0032-0006.jpg",
        "qa_status": "Correction Required",
        "error_type": "Missing Region",
        "severity": "High",
        "review_note": "Upper and left leaves are not covered by baseline polygon."
    }
]

qa_review_df = pd.DataFrame(qa_review_records)

print("=== AL-08 Human QA Review Log ===")
print(qa_review_df.to_string(index=False))

print("\n=== QA Status Summary ===")
print(qa_review_df["qa_status"].value_counts())

print("\n=== Error Type Summary ===")
print(qa_review_df["error_type"].value_counts())

# CSV 저장
qa_review_path = (
    PROJECT_ROOT
    / "outputs/auto_labeling/annotations"
    / "human_qa_review_log.csv"
)

qa_review_df.to_csv(
    qa_review_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved:", qa_review_path)


In [ ]:

# AL-08-03-01: Before/After Annotation Correction
# Baseline vs Experiment B 개선 마스크 비교

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

target_filename = "ave-0032-0006.jpg"
stem = Path(target_filename).stem

image_path = (
    PROJECT_ROOT / "data/raw/images" / target_filename
)

mask_dir = (
    PROJECT_ROOT / "outputs/auto_labeling/masks"
)

baseline_path = mask_dir / f"{stem}_pred_mask.png"
improved_path = mask_dir / f"{stem}_improved_mask.png"

# 저장된 파일을 다시 읽어 실행 순서 의존성 최소화
image = np.array(
    Image.open(image_path).convert("RGB")
)

baseline_mask = np.array(
    Image.open(baseline_path).convert("L")
) > 0

improved_mask = np.array(
    Image.open(improved_path).convert("L")
) > 0

assert baseline_mask.shape == improved_mask.shape
assert baseline_mask.shape == image.shape[:2]

# 추가된 영역과 기존 영역 구분
retained_region = baseline_mask & improved_mask
added_region = improved_mask & ~baseline_mask
removed_region = baseline_mask & ~improved_mask

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Baseline
axes[0].imshow(image)
axes[0].imshow(
    np.ma.masked_where(~baseline_mask, baseline_mask),
    cmap="Greens",
    alpha=0.5
)
axes[0].set_title("Baseline Mask")

# Experiment B
axes[1].imshow(image)
axes[1].imshow(
    np.ma.masked_where(~improved_mask, improved_mask),
    cmap="Greens",
    alpha=0.5
)
axes[1].set_title("Experiment B | Improved Mask")

# Change Map
change_map = np.zeros((*baseline_mask.shape, 4), dtype=float)

# 기존 유지 영역: 초록색
change_map[retained_region] = [0, 1, 0, 0.45]

# 새로 추가된 영역: 파란색
change_map[added_region] = [0, 0.4, 1, 0.55]

# 제거된 영역: 빨간색
change_map[removed_region] = [1, 0, 0, 0.55]

axes[2].imshow(image)
axes[2].imshow(change_map)
axes[2].set_title("Correction Change Map")

for ax in axes:
    ax.set_xlim(1150, 1500)
    ax.set_ylim(900, 400)
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

print("=== AL-08 Correction Comparison ===")
print("Baseline Pixels:", int(baseline_mask.sum()))
print("Improved Pixels:", int(improved_mask.sum()))
print("Retained Pixels:", int(retained_region.sum()))
print("Added Pixels:", int(added_region.sum()))
print("Removed Pixels:", int(removed_region.sum()))


In [ ]:

# AL-08-03-02: Third Point Prompt Setup
# 왼쪽 잎을 대상으로 추가 프롬프트 좌표 검증

import matplotlib.pyplot as plt
import numpy as np

# 기존 AL-08-03-01의 image 사용
correction_prompts = {
    "P1_Upper": (1320, 560),
    "P2_Lower": (1320, 700),
    "P3_Left": (1180, 575)
}

fig, ax = plt.subplots(figsize=(8, 10))

ax.imshow(image)

colors = {
    "P1_Upper": "red",
    "P2_Lower": "blue",
    "P3_Left": "yellow"
}

for label, (x, y) in correction_prompts.items():

    ax.scatter(
        x, y,
        c=colors[label],
        s=200,
        marker="*",
        edgecolors="black",
        linewidths=1.2
    )

    ax.annotate(
        label,
        (x, y),
        xytext=(12, -12),
        textcoords="offset points",
        color=colors[label],
        fontsize=11,
        fontweight="bold",
        bbox={
            "facecolor": "black",
            "alpha": 0.65,
            "pad": 3
        }
    )

ax.set_xlim(1130, 1500)
ax.set_ylim(900, 400)
ax.set_title("AL-08 Third Point Prompt Verification")
ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()

print("=== AL-08 Correction Prompt Setup ===")
for label, point in correction_prompts.items():
    print(f"{label}: {point}")


In [ ]:

# AL-08-03-03: Third Prompt Inference & QA
# Experiment B + P3_Left -> Experiment C

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from mobile_sam import SamPredictor

# AL-08-03-01에서 불러온 image, improved_mask 사용
# AL-08-03-02에서 정의한 correction_prompts 사용

predictor_correction = SamPredictor(sam)
predictor_correction.set_image(image)

# 1. P3 단독 추론
p3_x, p3_y = correction_prompts["P3_Left"]

masks, scores, _ = predictor_correction.predict(
    point_coords=np.array([[p3_x, p3_y]]),
    point_labels=np.array([1]),
    multimask_output=True
)

best_idx = int(np.argmax(scores))
p3_mask = masks[best_idx].astype(bool)
p3_score = float(scores[best_idx])

# 2. Experiment B와 통합
experiment_c_mask = improved_mask | p3_mask

# 3. 새로 추가된 영역
new_region = experiment_c_mask & ~improved_mask

# 4. 기존과 동일한 잠정 GT 평가
reference_path = (
    PROJECT_ROOT
    / "data/raw/labelIds/CropAndWeed"
    / "ave-0032-0006.png"
)

reference_mask = np.array(
    Image.open(reference_path).convert("L")
)

valid_pixels = reference_mask != 255
gt_binary = (reference_mask != 0) & valid_pixels

def evaluate_correction(mask):
    tp = int(np.sum(mask & gt_binary & valid_pixels))
    fp = int(np.sum(mask & ~gt_binary & valid_pixels))
    fn = int(np.sum(~mask & gt_binary & valid_pixels))

    iou = tp / (tp + fp + fn) if tp + fp + fn else np.nan
    dice = 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else np.nan
    precision = tp / (tp + fp) if tp + fp else np.nan
    recall = tp / (tp + fn) if tp + fn else np.nan

    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "iou": round(iou, 4),
        "dice": round(dice, 4),
        "precision": round(precision, 4),
        "recall": round(recall, 4)
    }

# 5. 비교 결과
comparison_c_df = pd.DataFrame([
    {
        "experiment": "Baseline",
        **evaluate_correction(baseline_mask)
    },
    {
        "experiment": "Experiment B",
        **evaluate_correction(improved_mask)
    },
    {
        "experiment": "Experiment C",
        **evaluate_correction(experiment_c_mask)
    }
])

print("=== AL-08 Prompt Correction Comparison ===")
print(comparison_c_df.to_string(index=False))

print("\n=== Added Region Summary ===")
print("P3 Predicted Pixels:", int(p3_mask.sum()))
print("Newly Added Pixels:", int(new_region.sum()))
print("Experiment C Pixels:", int(experiment_c_mask.sum()))
print("P3 Predicted Score:", round(p3_score, 4))

# 6. 시각화
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

display_masks = [
    ("Experiment B", improved_mask),
    ("P3 Left Only", p3_mask),
    ("Experiment C", experiment_c_mask)
]

for ax, (title, mask) in zip(axes, display_masks):

    ax.imshow(image)
    ax.imshow(
        np.ma.masked_where(~mask, mask),
        cmap="Greens",
        alpha=0.5,
        vmin=0,
        vmax=1
    )

    ax.set_title(title)
    ax.set_xlim(1130, 1500)
    ax.set_ylim(900, 400)
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()


In [ ]:

# AL-08-03-04-01: Positive + Negative Prompt Experiment
# Experiment D: 왼쪽 녹색 잎 선택적 분할

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Positive: 포함할 왼쪽 녹색 잎
# Negative: 인접한 붉은색 잎 (실험적 제외 지점)
point_coords = np.array([
    [1180, 575],  # Positive
    [1180, 525],  # Negative
], dtype=np.float32)

point_labels = np.array([1, 0], dtype=np.int32)

# MobileSAM 추론
predictor_correction.set_image(image)

masks_d, scores_d, _ = predictor_correction.predict(
    point_coords=point_coords,
    point_labels=point_labels,
    multimask_output=True
)

best_idx_d = int(np.argmax(scores_d))

p3_refined_mask = masks_d[best_idx_d].astype(bool)

# Experiment B와 통합
experiment_d_mask = improved_mask | p3_refined_mask

# 기존 잠정 평가 함수 재사용
comparison_d_df = pd.DataFrame([
    {
        "experiment": "Experiment B",
        **evaluate_correction(improved_mask)
    },
    {
        "experiment": "Experiment C",
        **evaluate_correction(experiment_c_mask)
    },
    {
        "experiment": "Experiment D",
        **evaluate_correction(experiment_d_mask)
    }
])

print("=== AL-08 Negative Prompt Comparison ===")
print(comparison_d_df.to_string(index=False))

print("\n=== Experiment D Summary ===")
print("Refined P3 Pixels:", int(p3_refined_mask.sum()))
print("Combined Pixels:", int(experiment_d_mask.sum()))
print("Model Estimated Score:", round(float(scores_d[best_idx_d]), 4))

# 시각화
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

display_masks = [
    ("Experiment B", improved_mask),
    ("Refined P3", p3_refined_mask),
    ("Experiment D", experiment_d_mask)
]

for ax, (title, mask) in zip(axes, display_masks):
    ax.imshow(image)

    ax.imshow(
        np.ma.masked_where(~mask, mask),
        cmap="Greens",
        alpha=0.5,
        vmin=0,
        vmax=1
    )

    ax.set_title(title)
    ax.set_xlim(1130, 1500)
    ax.set_ylim(900, 400)
    ax.grid(alpha=0.2)

# Positive / Negative 위치 표시
axes[1].scatter(
    point_coords[0, 0], point_coords[0, 1],
    c="lime", marker="*", s=180,
    edgecolors="black"
)

axes[1].scatter(
    point_coords[1, 0], point_coords[1, 1],
    c="red", marker="x", s=150,
    linewidths=3
)

plt.tight_layout()
plt.show()


In [ ]:

# AL-08-03-05: Multi-Mask Candidate Diagnosis

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

candidate_records = []

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for idx in range(len(masks_d)):

    candidate_mask = masks_d[idx].astype(bool)

    # Experiment B와 통합한 결과
    combined_mask = improved_mask | candidate_mask

    metrics = evaluate_correction(combined_mask)

    candidate_records.append({
        "candidate": idx,
        "predicted_score": round(float(scores_d[idx]), 4),
        "candidate_pixels": int(candidate_mask.sum()),
        "combined_pixels": int(combined_mask.sum()),
        "iou": metrics["iou"],
        "precision": metrics["precision"],
        "recall": metrics["recall"]
    })

    ax = axes[idx]
    ax.imshow(image)

    ax.imshow(
        np.ma.masked_where(~candidate_mask, candidate_mask),
        cmap="Greens",
        alpha=0.5,
        vmin=0,
        vmax=1
    )

    ax.set_title(
        f"Candidate {idx}\n"
        f"Pixels: {candidate_mask.sum():,} | "
        f"Score: {scores_d[idx]:.4f}"
    )

    ax.set_xlim(1130, 1500)
    ax.set_ylim(900, 400)
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

candidate_qa_df = pd.DataFrame(candidate_records)

print("=== AL-08 Multi-Mask Candidate QA ===")
print(candidate_qa_df.to_string(index=False))

print("\nSmallest Candidate:")
print(
    candidate_qa_df.loc[
        candidate_qa_df["candidate_pixels"].idxmin()
    ].to_string()
)


In [ ]:

# AL-08-03-06: Save Experiment E Correction Candidate

import numpy as np
import pandas as pd
from PIL import Image

# Candidate 0 + Experiment B
candidate_0_mask = masks_d[0].astype(bool)
experiment_e_mask = improved_mask | candidate_0_mask

# 저장 경로
corrected_mask_path = (
    PROJECT_ROOT
    / "outputs/auto_labeling/masks"
    / "ave-0032-0006_experiment_e_mask.png"
)

# Binary PNG 저장
Image.fromarray(
    (experiment_e_mask.astype(np.uint8) * 255)
).save(corrected_mask_path)

# 재로드 검증
reloaded_mask = np.array(
    Image.open(corrected_mask_path).convert("L")
) > 0

assert np.array_equal(
    experiment_e_mask,
    reloaded_mask
)

# 성능 기록
experiment_e_metrics = evaluate_correction(
    experiment_e_mask
)

experiment_e_record = {
    "image_name": "ave-0032-0006.jpg",
    "experiment": "Experiment E",
    "method": "Positive-Negative Prompt Candidate Selection",
    "selected_candidate": 0,
    "predicted_score": float(scores_d[0]),
    "mask_pixels": int(experiment_e_mask.sum()),
    **experiment_e_metrics,
    "qa_status": "pending_human_review",
    "evaluation_status": "provisional"
}

experiment_e_df = pd.DataFrame(
    [experiment_e_record]
)

metrics_path_e = (
    PROJECT_ROOT
    / "outputs/auto_labeling/annotations"
    / "experiment_e_correction_metrics.csv"
)

experiment_e_df.to_csv(
    metrics_path_e,
    index=False,
    encoding="utf-8-sig"
)

print("=== AL-08 Experiment E Export ===")
print("Mask Saved:", corrected_mask_path.exists())
print("Metrics Saved:", metrics_path_e.exists())
print("Mask Pixels:", int(experiment_e_mask.sum()))
print("IoU:", experiment_e_metrics["iou"])
print("Dice:", experiment_e_metrics["dice"])
print("Precision:", experiment_e_metrics["precision"])
print("Recall:", experiment_e_metrics["recall"])
print("Mask Reload Validation: PASS")


### AL-08-04. Before/After Polygon QA

기존 Baseline과 수정 후보 Experiment E를 Polygon으로 변환하여,
수정 전후의 어노테이션 형태와 변환 품질을 비교한다.

In [ ]:

# AL-08-04-01: Before/After Polygon QA

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

EPSILON_RATIO = 0.001
MIN_AREA = 100

def convert_mask_to_polygons(mask):
    binary = (mask > 0).astype(np.uint8) * 255

    contours, _ = cv2.findContours(
        binary,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    polygons = []

    for contour in contours:
        if cv2.contourArea(contour) < MIN_AREA:
            continue

        perimeter = cv2.arcLength(contour, True)

        simplified = cv2.approxPolyDP(
            contour,
            EPSILON_RATIO * perimeter,
            True
        )

        if len(simplified) >= 3:
            polygons.append(simplified)

    reconstructed = np.zeros_like(binary)

    if polygons:
        cv2.fillPoly(
            reconstructed,
            polygons,
            color=255
        )

    original_bool = binary > 0
    reconstructed_bool = reconstructed > 0

    intersection = np.logical_and(
        original_bool, reconstructed_bool
    ).sum()

    union = np.logical_or(
        original_bool, reconstructed_bool
    ).sum()

    conversion_iou = (
        intersection / union if union else np.nan
    )

    return polygons, conversion_iou

# 기존 Baseline과 Experiment E 비교
comparison_masks = {
    "Baseline": baseline_mask,
    "Experiment E": experiment_e_mask
}

polygon_comparison_records = []
correction_polygons = {}

fig, axes = plt.subplots(1, 2, figsize=(15, 7))

for ax, (name, mask) in zip(
    axes,
    comparison_masks.items()
):
    polygons, conversion_iou = convert_mask_to_polygons(mask)

    correction_polygons[name] = polygons

    display_image = image.copy()

    if polygons:
        cv2.drawContours(
            display_image,
            polygons,
            -1,
            (255, 0, 0),
            2
        )

    ax.imshow(display_image)
    ax.set_xlim(1130, 1500)
    ax.set_ylim(900, 400)
    ax.set_title(
        f"{name}\n"
        f"Polygons: {len(polygons)} | "
        f"Conversion IoU: {conversion_iou:.4f}"
    )
    ax.grid(alpha=0.2)

    polygon_comparison_records.append({
        "experiment": name,
        "mask_pixels": int(mask.sum()),
        "polygon_count": len(polygons),
        "total_vertices": sum(
            len(polygon) for polygon in polygons
        ),
        "conversion_iou": round(conversion_iou, 6)
    })

plt.tight_layout()
plt.show()

polygon_comparison_df = pd.DataFrame(
    polygon_comparison_records
)

print("=== AL-08 Before/After Polygon QA ===")
print(polygon_comparison_df.to_string(index=False))


In [ ]:

# AL-08-05-01: Corrected Annotation Export

import json
import numpy as np
import pandas as pd
from pathlib import Path

annotation_dir = (
    PROJECT_ROOT
    / "outputs/auto_labeling/annotations"
)

annotation_dir.mkdir(parents=True, exist_ok=True)

target_filename = "ave-0032-0006.jpg"

# AL-08-04에서 생성한 Experiment E Polygon
experiment_e_polygons = correction_polygons["Experiment E"]

height, width = experiment_e_mask.shape

corrected_annotations = []

for polygon_id, contour in enumerate(
    experiment_e_polygons, start=1
):
    points = contour.reshape(-1, 2).tolist()

    corrected_annotations.append({
        "polygon_id": polygon_id,
        "points": points,
        "vertex_count": len(points),
        "label": "plant_foreground_unverified",
        "source": "MobileSAM",
        "correction_method": "human_guided_prompt_selection"
    })

export_data_e = {
    "project": "Smart Agriculture Data Labeling",
    "image_name": target_filename,
    "width": width,
    "height": height,
    "model": "MobileSAM",
    "experiment": "Experiment E",
    "annotation_type": "polygon",
    "epsilon_ratio": 0.001,
    "min_area": 100,
    "qa_status": "pending_human_review",
    "evaluation_status": "provisional",
    "polygons": corrected_annotations
}

corrected_json_path = (
    annotation_dir
    / "experiment_e_corrected_annotations.json"
)

with open(corrected_json_path, "w", encoding="utf-8") as f:
    json.dump(
        export_data_e,
        f,
        ensure_ascii=False,
        indent=2
    )

# 수정 이력 기록
baseline_metrics = evaluate_correction(baseline_mask)
corrected_metrics = evaluate_correction(experiment_e_mask)

correction_history = pd.DataFrame([{
    "image_name": target_filename,
    "baseline_iou": baseline_metrics["iou"],
    "corrected_iou": corrected_metrics["iou"],
    "iou_delta": round(
        corrected_metrics["iou"] - baseline_metrics["iou"], 4
    ),
    "baseline_polygon_count": len(
        correction_polygons["Baseline"]
    ),
    "corrected_polygon_count": len(
        experiment_e_polygons
    ),
    "correction_method": "human_guided_prompt_selection",
    "qa_status": "pending_human_review"
}])

history_path = (
    annotation_dir / "annotation_correction_history.csv"
)

correction_history.to_csv(
    history_path,
    index=False,
    encoding="utf-8-sig"
)

print("=== AL-08 Corrected Annotation Export ===")
print("JSON Saved:", corrected_json_path.exists())
print("History Saved:", history_path.exists())
print("Corrected Polygons:", len(corrected_annotations))
print("Total Vertices:", sum(
    p["vertex_count"] for p in corrected_annotations
))
print("\n=== Correction History ===")
print(correction_history.to_string(index=False))


In [ ]:

# AL-08-05-02: Corrected Annotation Validation

import json
import pandas as pd

with open(corrected_json_path, "r", encoding="utf-8") as f:
    saved_corrected = json.load(f)

saved_history = pd.read_csv(history_path)

assert saved_corrected["experiment"] == "Experiment E"
assert saved_corrected["qa_status"] == "pending_human_review"
assert len(saved_corrected["polygons"]) == 3

for polygon in saved_corrected["polygons"]:
    points = polygon["points"]

    assert len(points) >= 3
    assert polygon["vertex_count"] == len(points)

    for x, y in points:
        assert 0 <= x < saved_corrected["width"]
        assert 0 <= y < saved_corrected["height"]

assert len(saved_history) == 1
assert saved_history.loc[0, "corrected_iou"] > (
    saved_history.loc[0, "baseline_iou"]
)

print("=== AL-08 Corrected Annotation Validation ===")
print("Image:", saved_corrected["image_name"])
print("Polygon Count:", len(saved_corrected["polygons"]))
print("Correction History Rows:", len(saved_history))
print("Coordinate Validation: PASS")
print("IoU Improvement Validation: PASS")
print("\nAL-08 Export Validation: PASS")


## AL-09. Final Quality Assurance

### AL-09-01. Performance Summary

MobileSAM 다중 이미지 파일럿의 성능 평가 결과와 Human QA 기록을 통합한다.
Baseline 성능과 수정 후보 성능을 구분하며, Label ID 매핑이 검증되지 않은
평가 결과는 Provisional(잠정) 상태로 유지한다.

In [ ]:

# AL-09-01-01: Final QA Data Integration
# 기존 성능 CSV와 Human QA 기록 통합

from pathlib import Path
import pandas as pd

OUTPUT_DIR = PROJECT_ROOT / "outputs/auto_labeling"
ANNOTATION_DIR = OUTPUT_DIR / "annotations"

# 1. 저장된 CSV 목록 확인
csv_files = sorted(OUTPUT_DIR.rglob("*.csv"))

print("=== Available QA CSV Files ===")
for path in csv_files:
    print(path.relative_to(PROJECT_ROOT))

# 2. 다중 이미지 평가 CSV 탐색
evaluation_candidates = []

for path in csv_files:
    try:
        columns = set(
            pd.read_csv(path, nrows=0).columns
        )

        required = {
            "image_name", "iou", "dice",
            "precision", "recall"
        }

        if required.issubset(columns):
            df = pd.read_csv(path)

            if len(df) == 5:
                evaluation_candidates.append(path)

    except (OSError, ValueError, pd.errors.ParserError):
        continue

print("\n=== Evaluation CSV Candidates ===")
for path in evaluation_candidates:
    print(path.relative_to(PROJECT_ROOT))

# 3. 후보가 하나일 때만 자동 선택
if len(evaluation_candidates) != 1:
    print(
        "\n확인 필요: 평가 CSV 후보가",
        len(evaluation_candidates),
        "개 발견되었습니다."
    )
    print(
        "파일명과 컬럼을 확인한 후 "
        "다음 단계에서 선택합니다."
    )
else:
    evaluation_path = evaluation_candidates[0]

    evaluation_df = pd.read_csv(evaluation_path)

    review_path = (
        ANNOTATION_DIR / "human_qa_review_log.csv"
    )
    history_path = (
        ANNOTATION_DIR / "annotation_correction_history.csv"
    )

    review_df = pd.read_csv(review_path)
    history_df = pd.read_csv(history_path)

    # 4. 이미지별 Baseline 성능과 QA 기록 통합
    final_qa_df = evaluation_df.merge(
        review_df[
            [
                "image_name",
                "qa_status",
                "error_type",
                "severity"
            ]
        ],
        on="image_name",
        how="left",
        validate="one_to_one"
    )

    # 5. 수정 전후 성능 추가
    correction_columns = [
        "image_name",
        "corrected_iou",
        "iou_delta",
        "correction_method"
    ]

    final_qa_df = final_qa_df.merge(
        history_df[correction_columns],
        on="image_name",
        how="left",
        validate="one_to_one"
    )

    final_qa_df["evaluation_status"] = "provisional"

    # 6. Final QA Summary 저장
    summary_path = (
        ANNOTATION_DIR / "final_qa_summary.csv"
    )

    final_qa_df.to_csv(
        summary_path,
        index=False,
        encoding="utf-8-sig"
    )

    print("\n=== AL-09 Final QA Summary ===")
    print(final_qa_df.to_string(index=False))

    print("\n=== Final QA Integration ===")
    print("Images:", len(final_qa_df))
    print("QA Records:", final_qa_df["qa_status"].notna().sum())
    print(
        "Correction Records:",
        final_qa_df["corrected_iou"].notna().sum()
    )
    print("Saved:", summary_path)


### AL-09-02. Annotation Integrity Check

저장된 Polygon JSON과 QA CSV의 구조, 좌표 유효성,
이미지별 Polygon 수 및 수정 후보의 검수 상태를 확인한다.

본 검증은 파일 무결성 검사이며, 라벨의 의미적 정확성이나
Human QA 최종 승인을 의미하지 않는다.

In [ ]:

# AL-09-02-01: Annotation Integrity Validation

import json
import pandas as pd
from pathlib import Path

annotation_dir = (
    PROJECT_ROOT / "outputs/auto_labeling/annotations"
)

original_path = (
    annotation_dir / "mobile_sam_polygon_annotations.json"
)

corrected_path = (
    annotation_dir / "experiment_e_corrected_annotations.json"
)

polygon_qa_path = (
    annotation_dir / "mobile_sam_polygon_qa.csv"
)

final_qa_path = (
    annotation_dir / "final_qa_summary.csv"
)

# 1. 파일 존재 확인
required_files = [
    original_path,
    corrected_path,
    polygon_qa_path,
    final_qa_path
]

for path in required_files:
    assert path.exists(), f"Missing file: {path}"

# 2. JSON 및 CSV 로드
with open(original_path, encoding="utf-8") as f:
    original_data = json.load(f)

with open(corrected_path, encoding="utf-8") as f:
    corrected_data = json.load(f)

polygon_qa = pd.read_csv(polygon_qa_path)
final_qa = pd.read_csv(final_qa_path)

# 3. 원본 JSON 구조 확인
print("=== Original JSON Structure ===")
print("Root Type:", type(original_data).__name__)

if isinstance(original_data, dict):
    print("Root Keys:", list(original_data.keys()))
elif isinstance(original_data, list):
    print("Records:", len(original_data))

# 4. 수정 JSON 검증
assert corrected_data["image_name"] == "ave-0032-0006.jpg"
assert corrected_data["qa_status"] == "pending_human_review"

corrected_polygons = corrected_data["polygons"]

assert len(corrected_polygons) == 3

height = corrected_data["height"]
width = corrected_data["width"]

for polygon in corrected_polygons:
    points = polygon["points"]

    assert len(points) >= 3
    assert polygon["vertex_count"] == len(points)

    for x, y in points:
        assert 0 <= x < width
        assert 0 <= y < height

# 5. CSV 무결성
assert len(polygon_qa) == 5
assert len(final_qa) == 5

assert final_qa["image_name"].is_unique
assert polygon_qa["image_name"].is_unique

assert set(final_qa["image_name"]) == set(
    polygon_qa["image_name"]
)

# 6. Polygon QA 결과
print("\n=== Polygon QA Records ===")
print(
    polygon_qa[
        [
            "image_name",
            "polygon_count",
            "total_vertices",
            "conversion_iou"
        ]
    ].to_string(index=False)
)

# 7. 수정 후보 결과
print("\n=== Corrected Annotation ===")
print("Image:", corrected_data["image_name"])
print("Polygon Count:", len(corrected_polygons))
print(
    "Total Vertices:",
    sum(p["vertex_count"] for p in corrected_polygons)
)
print("QA Status:", corrected_data["qa_status"])

print("\nAL-09-02-01 Structural Validation: PASS")


In [ ]:

# AL-09-02-02: Inspect Original Annotation Structure

annotations = original_data["annotations"]

print("=== Original Annotation Structure ===")
print("Annotations Type:", type(annotations).__name__)
print("Annotation Records:", len(annotations))

if isinstance(annotations, list) and annotations:
    first_record = annotations[0]

    print("\n=== First Annotation Record ===")
    print("Record Type:", type(first_record).__name__)

    if isinstance(first_record, dict):
        print("Record Keys:", list(first_record.keys()))

        for key, value in first_record.items():
            if isinstance(value, list):
                print(
                    f"{key}: list, "
                    f"length={len(value)}"
                )

                if value and isinstance(value[0], dict):
                    print(
                        f"  First Item Keys: "
                        f"{list(value[0].keys())}"
                    )
            else:
                print(f"{key}: {value}")

elif isinstance(annotations, dict):
    print("Annotation Keys:", list(annotations.keys())[:10])

    first_key = next(iter(annotations), None)

    if first_key is not None:
        first_record = annotations[first_key]

        print("\n=== First Annotation Record ===")
        print("Key:", first_key)
        print("Record Type:", type(first_record).__name__)

        if isinstance(first_record, dict):
            print("Record Keys:", list(first_record.keys()))

print("\nAL-09-02-02 Structure Inspection: COMPLETE")


In [ ]:

# AL-09-02-03: Full Annotation Integrity Validation

import numpy as np
import pandas as pd

annotations = original_data["annotations"]

validation_records = []
validation_errors = []

# QA CSV를 이미지 이름으로 조회
qa_lookup = polygon_qa.set_index("image_name")

# 이미지 목록 및 중복 검사
json_names = [item["image_name"] for item in annotations]
csv_names = polygon_qa["image_name"].tolist()

assert len(json_names) == len(set(json_names)), (
    "Duplicate image names in JSON"
)
assert set(json_names) == set(csv_names), (
    "JSON and CSV image lists do not match"
)

for item in annotations:

    filename = item["image_name"]
    width = item["width"]
    height = item["height"]
    polygons = item["polygons"]

    errors = []

    if item["annotation_type"] != "polygon":
        errors.append("Invalid annotation type")

    if item["qa_status"] != "pending_human_review":
        errors.append("Unexpected QA status")

    if item["label"] != "plant_foreground_unverified":
        errors.append("Unexpected label")

    if width <= 0 or height <= 0:
        errors.append("Invalid image dimensions")

    polygon_ids = [
        polygon["polygon_id"] for polygon in polygons
    ]

    if len(polygon_ids) != len(set(polygon_ids)):
        errors.append("Duplicate polygon IDs")

    total_vertices = 0

    for polygon in polygons:

        points = polygon["points"]
        vertex_count = polygon["vertex_count"]

        total_vertices += vertex_count

        if len(points) < 3:
            errors.append("Polygon has fewer than 3 points")
            continue

        if vertex_count != len(points):
            errors.append("Vertex count mismatch")

        coords = np.asarray(points)

        if coords.shape != (len(points), 2):
            errors.append("Invalid coordinate shape")
            continue

        if not np.issubdtype(coords.dtype, np.number):
            errors.append("Non-numeric coordinates")
            continue

        if not np.isfinite(coords).all():
            errors.append("Non-finite coordinates")
            continue

        if not (
            (coords[:, 0] >= 0).all()
            and (coords[:, 0] < width).all()
            and (coords[:, 1] >= 0).all()
            and (coords[:, 1] < height).all()
        ):
            errors.append("Coordinates out of bounds")

    # CSV와 비교
    expected_count = int(
        qa_lookup.loc[filename, "polygon_count"]
    )
    expected_vertices = int(
        qa_lookup.loc[filename, "total_vertices"]
    )

    if len(polygons) != expected_count:
        errors.append("Polygon count mismatch with CSV")

    if total_vertices != expected_vertices:
        errors.append("Vertex count mismatch with CSV")

    status = "PASS" if not errors else "FAIL"

    validation_records.append({
        "image_name": filename,
        "polygon_count": len(polygons),
        "total_vertices": total_vertices,
        "validation_status": status,
        "error_count": len(errors)
    })

    for error in errors:
        validation_errors.append({
            "image_name": filename,
            "error": error
        })

# 결과 정리
integrity_df = pd.DataFrame(validation_records)
integrity_errors_df = pd.DataFrame(validation_errors)

print("=== AL-09 Full Annotation Integrity ===")
print(integrity_df.to_string(index=False))

print("\n=== Integrity Summary ===")
print("Images:", len(integrity_df))
print("Total Polygons:", integrity_df["polygon_count"].sum())
print("Total Vertices:", integrity_df["total_vertices"].sum())
print("Passed Images:", (
    integrity_df["validation_status"] == "PASS"
).sum())
print("Failed Images:", (
    integrity_df["validation_status"] == "FAIL"
).sum())
print("Total Errors:", len(integrity_errors_df))

# 검증 결과 저장
integrity_path = (
    annotation_dir / "annotation_integrity_report.csv"
)

integrity_df.to_csv(
    integrity_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved:", integrity_path)

if len(integrity_errors_df) == 0:
    print("\nAL-09-02-03 Integrity Validation: PASS")
else:
    print("\n=== Validation Errors ===")
    print(integrity_errors_df.to_string(index=False))
    print("\nAL-09-02-03 Integrity Validation: FAIL")


### AL-09-03. Quality Gate Assessment

파일 무결성, Polygon 변환 품질, Segmentation 성능,
라벨 매핑 및 Human QA 승인 상태를 구분하여 평가한다.

Segmentation IoU 기준 0.80은 본 파일럿의 내부 검토 기준이며,
Reference Label 매핑이 미확인 상태이므로 잠정 평가로 관리한다.

In [ ]:

# AL-09-03-01: Image-Level Quality Gate Assessment

import numpy as np
import pandas as pd

IOU_THRESHOLD = 0.80
CONVERSION_THRESHOLD = 0.99

# AL-09-01, AL-09-02에서 생성한 CSV 로드
summary_df = pd.read_csv(
    annotation_dir / "final_qa_summary.csv"
)

polygon_df = pd.read_csv(
    annotation_dir / "mobile_sam_polygon_qa.csv"
)

integrity_df = pd.read_csv(
    annotation_dir / "annotation_integrity_report.csv"
)

# 이미지별 Quality Gate 데이터 통합
gate_df = summary_df[
    [
        "image_name",
        "iou",
        "corrected_iou",
        "qa_status"
    ]
].merge(
    polygon_df[
        ["image_name", "conversion_iou"]
    ],
    on="image_name",
    validate="one_to_one"
).merge(
    integrity_df[
        ["image_name", "validation_status"]
    ],
    on="image_name",
    validate="one_to_one"
)

# G1: 파일 무결성
gate_df["G1_integrity"] = np.where(
    gate_df["validation_status"] == "PASS",
    "PASS", "FAIL"
)

# G2: Polygon 변환 품질
gate_df["G2_conversion"] = np.where(
    gate_df["conversion_iou"] >= CONVERSION_THRESHOLD,
    "PASS", "REVIEW"
)

# G3: 잠정 Segmentation 성능
# 수정 후보는 별도로 평가하되 원본 결과를 덮어쓰지 않음
gate_df["G3_baseline"] = np.where(
    gate_df["iou"] >= IOU_THRESHOLD,
    "MEETS_PROVISIONAL_THRESHOLD",
    "BELOW_PROVISIONAL_THRESHOLD"
)

gate_df["G3_corrected_candidate"] = gate_df[
    "corrected_iou"
].apply(
    lambda x: (
        "NOT_APPLICABLE" if pd.isna(x)
        else (
            "MEETS_PROVISIONAL_THRESHOLD"
            if x >= IOU_THRESHOLD
            else "BELOW_PROVISIONAL_THRESHOLD"
        )
    )
)

# G4: Label ID 매핑 검증
gate_df["G4_label_mapping"] = "UNVERIFIED"

# G5: Human QA 최종 승인
gate_df["G5_human_approval"] = "PENDING"

# 최종 승인 상태
gate_df["final_release_status"] = "HOLD"

# 저장
gate_path = (
    annotation_dir / "quality_gate_assessment.csv"
)

gate_df.to_csv(
    gate_path,
    index=False,
    encoding="utf-8-sig"
)

print("=== AL-09 Quality Gate Assessment ===")
print(
    gate_df[
        [
            "image_name",
            "G1_integrity",
            "G2_conversion",
            "G3_baseline",
            "G3_corrected_candidate",
            "G4_label_mapping",
            "G5_human_approval",
            "final_release_status"
        ]
    ].to_string(index=False)
)

print("\n=== Quality Gate Summary ===")
print(
    "G1 PASS:",
    (gate_df["G1_integrity"] == "PASS").sum()
)
print(
    "G2 PASS:",
    (gate_df["G2_conversion"] == "PASS").sum()
)
print(
    "G3 Baseline Threshold Met:",
    (
        gate_df["G3_baseline"]
        == "MEETS_PROVISIONAL_THRESHOLD"
    ).sum()
)
print(
    "Final Release Approved:",
    (
        gate_df["final_release_status"] == "APPROVED"
    ).sum()
)

print("\nSaved:", gate_path)


### AL-09-04. Final QA Report

MobileSAM 파일럿의 성능, Polygon 변환 품질, 어노테이션
무결성, Human-Guided Correction 및 Quality Gate 결과를
통합하여 최종 QA 보고서를 생성한다.

최종 배포 승인과 포트폴리오 실험 완료 여부는 별도로 관리한다.

In [ ]:

# AL-09-04-01: Generate Final QA Report

from datetime import date
import pandas as pd

annotation_dir = (
    PROJECT_ROOT / "outputs/auto_labeling/annotations"
)

output_dir = PROJECT_ROOT / "outputs/auto_labeling"

summary = pd.read_csv(
    annotation_dir / "final_qa_summary.csv"
)

polygon_qa = pd.read_csv(
    annotation_dir / "mobile_sam_polygon_qa.csv"
)

integrity = pd.read_csv(
    annotation_dir / "annotation_integrity_report.csv"
)

gates = pd.read_csv(
    annotation_dir / "quality_gate_assessment.csv"
)

history = pd.read_csv(
    annotation_dir / "annotation_correction_history.csv"
)

# 1. 핵심 지표 계산
metric_columns = ["iou", "dice", "precision", "recall"]
means = summary[metric_columns].mean()

image_count = len(summary)
polygon_count = int(integrity["polygon_count"].sum())
vertex_count = int(integrity["total_vertices"].sum())

integrity_pass = int(
    (integrity["validation_status"] == "PASS").sum()
)

conversion_pass = int(
    (gates["G2_conversion"] == "PASS").sum()
)

threshold_pass = int(
    (gates["G3_baseline"]
     == "MEETS_PROVISIONAL_THRESHOLD").sum()
)

approved = int(
    (gates["final_release_status"] == "APPROVED").sum()
)

# 2. 수정 이력
correction_rows = []

for _, row in history.iterrows():
    correction_rows.append(
        f"| {row['image_name']} | "
        f"{row['baseline_iou']:.4f} | "
        f"{row['corrected_iou']:.4f} | "
        f"+{row['iou_delta']:.4f} | "
        f"{row['qa_status']} |"
    )

correction_table = "\n".join(correction_rows)

# 3. 이미지별 성능 표
performance_rows = []

for _, row in summary.iterrows():
    performance_rows.append(
        f"| {row['image_name']} | "
        f"{row['iou']:.4f} | "
        f"{row['dice']:.4f} | "
        f"{row['precision']:.4f} | "
        f"{row['recall']:.4f} | "
        f"{row['qa_status']} |"
    )

performance_table = "\n".join(performance_rows)

# 4. 보고서 작성
report = f"""# Smart Agriculture Labeling — Final QA Report

Generated: {date.today().isoformat()}

## 1. Project Overview

- Model: MobileSAM
- Task: AI-Assisted Plant Foreground Segmentation
- Pilot Images: {image_count}
- Annotation Format: Polygon JSON
- Evaluation Status: Provisional
- Final Release Status: HOLD

## 2. Baseline Performance

| Metric | Mean |
|---|---:|
| IoU | {means['iou']:.4f} |
| Dice | {means['dice']:.4f} |
| Precision | {means['precision']:.4f} |
| Recall | {means['recall']:.4f} |

위 지표는 5개 이미지의 단순 평균이며,
Reference Label ID 매핑이 검증되지 않은 잠정 결과이다.

## 3. Image-Level Performance

| Image | IoU | Dice | Precision | Recall | QA Status |
|---|---:|---:|---:|---:|---|
{performance_table}

## 4. Annotation Integrity

- Images: {image_count}
- Original Polygons: {polygon_count}
- Original Vertices: {vertex_count}
- Integrity PASS: {integrity_pass}/{image_count}
- Polygon Conversion PASS: {conversion_pass}/{image_count}
- Mean Conversion IoU: {polygon_qa['conversion_iou'].mean():.4f}

Conversion IoU는 Polygon이 원본 예측 마스크를
얼마나 충실하게 재현하는지 측정한다.
실제 Segmentation 정확도와는 구분된다.

## 5. Human-Guided Correction

| Image | Baseline IoU | Corrected IoU | Delta | QA Status |
|---|---:|---:|---:|---|
{correction_table}

수정은 Human-Guided Prompt Selection 방식으로 수행되었다.
수정 결과는 별도 후보 Annotation으로 저장되었으며,
최종 Human QA 승인을 받지 않았다.

## 6. Quality Gate Assessment

| Gate | Result |
|---|---|
| G1 File Integrity | {integrity_pass}/{image_count} PASS |
| G2 Polygon Conversion | {conversion_pass}/{image_count} PASS |
| G3 Provisional IoU >= 0.80 | {threshold_pass}/{image_count} |
| G4 Label Mapping | UNVERIFIED |
| G5 Human Approval | PENDING |
| Final Release | {approved}/{image_count} APPROVED |

IoU 0.80은 본 파일럿의 내부 검토 기준이다.

## 7. Limitations

1. Reference Label ID와 255의 의미가 최종 검증되지 않았다.
2. 평가 지표는 잠정 Reference Mask를 기준으로 산출되었다.
3. Point Prompt는 사람이 선택했으므로 완전 자동 라벨링이 아니다.
4. Polygon 구조 검증은 실제 경계 정확성을 보장하지 않는다.
5. Polygon Self-intersection 및 모든 세부 객체의 수동 검수는 미완료이다.
6. Experiment E는 최종 승인된 Annotation이 아닌 수정 후보이다.
7. 파일럿 이미지 5개만으로 일반화 성능을 판단할 수 없다.

## 8. Final Conclusion

MobileSAM 기반 AI 보조 라벨링 파이프라인을 구축하고,
다중 이미지 평가, Polygon 변환, 오류 분석,
Human-Guided Correction 및 Quality Gate 검증을 수행하였다.

기술적 산출물의 구조적 무결성은 검증되었으나,
의미적 라벨 검증과 최종 Human QA가 완료되지 않아
Annotation Release는 HOLD 상태로 유지한다.
"""

# 5. 보고서 저장
report_path = output_dir / "mobile_sam_final_qa_report.md"

report_path.write_text(
    report,
    encoding="utf-8"
)

# 6. 저장 검증
assert report_path.exists()
assert report_path.stat().st_size > 0

saved_report = report_path.read_text(encoding="utf-8")

assert "Final Release Status: HOLD" in saved_report
assert "Human-Guided Correction" in saved_report

print("=== AL-09 Final QA Report ===")
print("Saved:", report_path)
print("File Size:", report_path.stat().st_size, "bytes")
print("Images:", image_count)
print("Original Polygons:", polygon_count)
print("Mean Baseline IoU:", round(means["iou"], 4))
print("Mean Conversion IoU:",
      round(polygon_qa["conversion_iou"].mean(), 4))
print("Final Release Approved:", approved)

print("\nAL-09-04 Report Generation: PASS")
